In [8]:
import pandas as pd
import math
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
import matplotlib.pyplot as plt

In [27]:
df_casos = pd.read_csv("casos_talara.csv", low_memory=False)
df_clima = pd.read_csv("clima_talara.csv", low_memory=False)

In [28]:
df_train = pd.concat([df_casos.drop(["ano", "semana"], axis=1), df_clima], axis=1)
df_train.head()

,A97.0P,A97.1P,A97.2P,A97.0C,A97.1C,A97.2C,year,week,temperature_2m_mean,precipitation_sum,relative_humidity_2m_mean
0,0,0,0,0,0,0,2001.0,1.0,23.400000,0.5,74.000000
1,0,0,0,0,0,0,2001.0,2.0,23.142857,12.2,80.428571
2,0,0,0,0,0,0,2001.0,3.0,23.285714,18.7,82.142857
3,0,0,0,0,0,0,2001.0,4.0,24.585714,2.4,70.285714
4,0,0,0,0,0,0,2001.0,5.0,24.628571,3.2,75.142857


In [29]:
def get_weeks(mon):
  w_sin_cod = []
  w_cos_cod = []

  for i in range(len(mon)):
    w_sin_cod.append(math.sin(math.pi*mon.iloc[i]/53))
    w_cos_cod.append(math.sin(math.pi*mon.iloc[i]/53))

  return pd.DataFrame({"m_sin": w_sin_cod, "m_cos": w_cos_cod})

In [61]:
target = "A97.0P" # Casos probables de dengue sin signos de alarma

# Remover parámetros no usados, reemplazar valores de semana por sin & cos
weeks = get_weeks(df_train["week"])
df = df_train.drop(columns=["A97.0C", "A97.1P", "A97.1C", "A97.2P", "A97.2C", "week"])
df = pd.concat([weeks, df], axis=1)

xx = df.drop(columns=[target]).fillna(0)
yy = df[target].fillna(0)

In [92]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Separar datos: 80% entrenamiento, 20% prueba
x_train, x_test, y_train, y_test = train_test_split(
    xx, yy,
    test_size=0.2,
    random_state=42
)

# Grado del polinomio
n = 3

# Transformación polinómica
poly = PolynomialFeatures(degree=n)

x_train_poly = poly.fit_transform(x_train)
x_test_poly = poly.transform(x_test)

# Entrenar modelo
model = LinearRegression()
model.fit(x_train_poly, y_train)

# Predicciones
y_pred = model.predict(x_test_poly)

# Evaluación
print(f"R²: {r2_score(y_test, y_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")
print(f"RMSE: {mean_squared_error(y_test, y_pred) ** 0.5:.4f}")

R²: 0.1058
MAE: 6.3640
RMSE: 28.2225


In [77]:
print(len(x_train))
print(len(x_test))

1102
276


In [78]:
print(y_pred)

[ 6.27504457e+00  2.21642564e+00 -2.94552582e+00  1.42803383e+00
  4.39146068e+00 -1.10332436e+00  1.99607736e+01  2.22950937e+00
  9.35579160e+00  1.29970495e+01  7.73558453e-01 -8.13882013e-01
 -5.08059619e-01  3.73843407e-01 -7.34355808e-01 -1.78426116e+00
  8.00372191e-01  5.50165379e-01  2.26635785e+00  7.70452363e-01
  6.59246589e+00 -2.67171828e+00  7.37892916e-01  4.25748846e-01
  9.95979178e+00 -4.15547160e+00  1.43423795e-01 -1.93334985e-01
 -1.01441369e+00 -6.40487020e-01 -2.15821725e+00  6.70423077e+00
  3.12752155e+00  2.03919228e-01  1.74225154e-01  6.99123658e+00
  1.19017507e+00  1.65062199e+01  1.80193143e+00 -1.57321187e+00
  5.32615634e+00 -2.62273478e+00 -1.72007414e+00 -2.79919009e+00
  2.87278959e+00  7.14816167e+00  1.05235385e+00  3.56344969e+00
  1.09810530e+00  2.58900425e-01  3.27081595e+00 -9.99991803e-01
  1.16211535e+01  1.20511594e+00 -3.88803108e-01  1.27828795e+00
 -2.04348261e-01 -1.73906026e+00  3.54982563e+00 -2.15842182e+00
  4.07497782e+00 -2.37785